# FunctionGemma: a first look

**Goal:** see what an on-device function-calling model does *out of the box*, before we fine-tune it.

Model: `google/functiongemma-270m-it` (270M parameters, Gemma 3 270M fine-tuned for function calling, released Dec 2025)


In [ ]:
%%capture
!pip install -U transformers accelerate

## 1. Load the model


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# If the Google repo asks you to accept the Gemma license, either accept it on the
# Hugging Face page or use the mirror: "unsloth/functiongemma-270m-it"
MODEL_ID = "google/functiongemma-270m-it"

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto").to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2
print(f"device: {device} | dtype: {model.dtype}")
print(f"parameters: {n_params / 1e6:.0f}M | weights in memory: {size_mb:.0f} MB")

/home/lpozzi/Git/data-science-lectures/llm_lora/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/lpozzi/Git/data-science-lectures/llm_lora/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


device: cuda | dtype: torch.bfloat16
parameters: 268M | weights in memory: 511 MB


In [10]:
print(model)

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

In [2]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "set_alarm",
            "description": "Sets an alarm on the phone at a given time.",
            "parameters": {
                "type": "object",
                "properties": {
                    "time": {"type": "string", "description": "Alarm time in HH:MM 24h format"},
                    "label": {"type": "string", "description": "Optional label for the alarm"},
                },
                "required": ["time"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "send_message",
            "description": "Sends a text message to a contact.",
            "parameters": {
                "type": "object",
                "properties": {
                    "contact": {"type": "string", "description": "Name of the contact"},
                    "text": {"type": "string", "description": "Body of the message"},
                },
                "required": ["contact", "text"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "create_calendar_event",
            "description": "Creates an event in the calendar.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string", "description": "Title of the event"},
                    "date": {"type": "string", "description": "Date in YYYY-MM-DD format"},
                    "time": {"type": "string", "description": "Start time in HH:MM 24h format"},
                },
                "required": ["title", "date"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "toggle_setting",
            "description": "Turns a phone setting on or off.",
            "parameters": {
                "type": "object",
                "properties": {
                    "setting": {
                        "type": "string",
                        "enum": ["wifi", "bluetooth", "flashlight", "do_not_disturb"],
                        "description": "The setting to change",
                    },
                    "enabled": {"type": "boolean", "description": "true to turn on, false to turn off"},
                },
                "required": ["setting", "enabled"],
            },
        },
    },
]

## 3. What does the model actually see?
FunctionGemma needs a **developer message** that activates function calling.
The chat template injects the tool schemas into the prompt.
How many tokens do the tool definitions cost? On a phone, every prompt token is latency.

In [3]:
DEVELOPER_PROMPT = "You are a model that can do function calling with the following functions"

def build_messages(user_request):
    return [
        {"role": "developer", "content": DEVELOPER_PROMPT},
        {"role": "user", "content": user_request},
    ]

prompt_text = tokenizer.apply_chat_template(
    build_messages("Set an alarm for 7am"),
    tools=TOOLS,
    add_generation_prompt=True,
    tokenize=False,
)
print(prompt_text)
print("prompt tokens:", len(tokenizer(prompt_text, add_special_tokens=False)["input_ids"]))

<bos><start_of_turn>developer
You are a model that can do function calling with the following functions<start_function_declaration>declaration:set_alarm{description:<escape>Sets an alarm on the phone at a given time.<escape>,parameters:{properties:{label:{description:<escape>Optional label for the alarm<escape>,type:<escape>STRING<escape>},time:{description:<escape>Alarm time in HH:MM 24h format<escape>,type:<escape>STRING<escape>}},required:[<escape>time<escape>],type:<escape>OBJECT<escape>}}<end_function_declaration><start_function_declaration>declaration:send_message{description:<escape>Sends a text message to a contact.<escape>,parameters:{properties:{contact:{description:<escape>Name of the contact<escape>,type:<escape>STRING<escape>},text:{description:<escape>Body of the message<escape>,type:<escape>STRING<escape>}},required:[<escape>contact<escape>,<escape>text<escape>],type:<escape>OBJECT<escape>}}<end_function_declaration><start_function_declaration>declaration:create_calendar

## 4. Generate and parse a function call
The model does **not** return JSON. It emits its own format:
```
<start_function_call>call:set_alarm{time:<escape>07:00<escape>}<end_function_call>
```
Parsing it into a Python call is the app's job.

In [4]:
import re
import json
import time

CALL_RE = re.compile(r"<start_function_call>call:([\w\.]+)\{(.*?)\}<end_function_call>", re.S)
ARG_RE = re.compile(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]+))", re.S)


def parse_value(raw):
    raw = raw.strip()
    try:
        return json.loads(raw)  # numbers, true/false, null
    except json.JSONDecodeError:
        return raw


def parse_calls(text):
    """Extract [{"name": ..., "arguments": {...}}] from FunctionGemma output.
    Flat arguments only: nested objects/arrays are not handled."""
    calls = []
    for name, args_str in CALL_RE.findall(text):
        args = {}
        for key, escaped, bare in ARG_RE.findall(args_str):
            args[key] = escaped if escaped or not bare else parse_value(bare)
        calls.append({"name": name, "arguments": args})
    return calls


@torch.no_grad()
def ask(user_request, max_new_tokens=128):
    inputs = tokenizer.apply_chat_template(
        build_messages(user_request),
        tools=TOOLS,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)
    start = time.perf_counter()
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # deterministic: same request, same call
        pad_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.perf_counter() - start
    new_tokens = out[0][inputs["input_ids"].shape[-1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=False)
    return {"raw": raw, "calls": parse_calls(raw), "seconds": elapsed, "tokens": len(new_tokens)}


# r = ask("Set an alarm for 7am")
# print(r["raw"])
# print(r["calls"])
# print(f"{r['tokens']} tokens in {r['seconds']:.2f}s")

## 5. Calling a tool or not calling one

A function-calling model has to make **two** decisions:
1. **Should I call a function at all?**
2. If yes, **which one, and with which arguments?**

When FunctionGemma decides not to call anything, it writes **plain text** instead of a `<start_function_call>` block.
Let's see what it does with requests that should *not* trigger an action.

In [5]:
CANNOT_EXECUTE = {
    "type": "function",
    "function": {
        "name": "cannot_execute",
        "description": "Use this when no other function can fulfill the request.",
        "parameters": {
            "type": "object",
            "properties": {
                "reason": {
                    "type": "string",
                    "enum": ["missing_argument", "not_supported", "out_of_scope"],
                    "description": "Why the request cannot be executed",
                },
            },
            "required": ["reason"],
        },
    },
}

TOOLS_WITH_FALLBACK = TOOLS + [CANNOT_EXECUTE]

@torch.no_grad()
def ask(user_request, tools=TOOLS, max_new_tokens=128):
    inputs = tokenizer.apply_chat_template(
        build_messages(user_request),
        tools=tools,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)
    start = time.perf_counter()
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        top_p=None,
        top_k=None,
        pad_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.perf_counter() - start
    new_tokens = out[0][inputs["input_ids"].shape[-1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=False)
    return {"raw": raw, "calls": parse_calls(raw), "seconds": elapsed, "tokens": len(new_tokens)}

NO_CALL_REQUESTS = [
    # missing required argument: which contact? what text?
    "Send a message",
    # an action the app does not have
    "Order a pizza for tonight",
    # a question, not a command
    "What's the capital of France?",
    # chit-chat
    "Thanks, that's all!",
]

import pandas as pd

def short(r):
    if r["calls"]:
        return "; ".join(f"{c['name']}({c['arguments']})" for c in r["calls"])
    return "TEXT: " + r["raw"].replace("<end_of_turn>", "").strip()[:60]

# should call a real function / should NOT call a real function
TEST_REQUESTS = [(req, "no call") for req in NO_CALL_REQUESTS]

rows = []
for req, expected in TEST_REQUESTS:
    rows.append({
        "request": req,
        "expected": expected,
        "without fallback": short(ask(req, tools=TOOLS)),
        "with fallback": short(ask(req, tools=TOOLS_WITH_FALLBACK)),
    })

comparison = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 70)
comparison

,request,expected,without fallback,with fallback
0,Send a message,no call,TEXT: I can help with sending messages. Could you please specify t,"TEXT: I can certainly help with sending messages. To proceed, coul"
1,Order a pizza for tonight,no call,"TEXT: I am sorry, but I cannot assist with ordering pizza. My curr","TEXT: I apologize, but I cannot assist with ordering or managing c"
2,What's the capital of France?,no call,"TEXT: I apologize, but I cannot assist with finding the capital of","TEXT: I apologize, but I cannot assist with finding the capital of"
3,"Thanks, that's all!",no call,TEXT: I'm glad to assist. How can I help you today?,"TEXT: I apologize, but I cannot assist with managing or delegating"


In [7]:
import pandas as pd

def short(r):
    if r["calls"]:
        return "; ".join(f"{c['name']}({c['arguments']})" for c in r["calls"])
    return "TEXT: " + r["raw"].replace("<end_of_turn>", "").strip()[:60]

# should call a real function / should NOT call a real function
TEST_REQUESTS = [(req, "call") for req in REQUESTS[:8]] + [(req, "no call") for req in NO_CALL_REQUESTS]

rows = []
for req, expected in NO_CALL_REQUESTS:
    rows.append({
        "request": req,
        "expected": expected,
        "without fallback": short(ask(req, tools=TOOLS)),
        "with fallback": short(ask(req, tools=TOOLS_WITH_FALLBACK)),
    })

comparison = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 70)
comparison

NameError: name 'REQUESTS' is not defined

### What did we see?
- The model **can** decide not to call a function: in all four cases it answered in text.
- For the missing argument it even **asked a follow-up question** ("Could you please specify the contact...?").
- But the text is **free-form and verbose**: the app can't reliably tell *why* nothing was called
  (missing argument? unsupported action? just a question?) without parsing sentences.
- These phone actions are close to what FunctionGemma was trained on. On our own app the picture will be different,
  as we'll see in the probe.

### Design choice: the model never writes replies to the user
A 270M model is good at short structured outputs, weak at writing text. So we split the work:

| model output | meaning | what the app does |
|---|---|---|
| a function call | valid command | executes it; if the object doesn't exist, shows a template ("No video called 'whisper'") |
| `missing_argument` | a required value is missing | asks a template question ("Which contact?") |
| `not_supported` | an action the app doesn't have | "I can't do that yet." |
| `out_of_scope` | a question or chit-chat | "I can only help with ..." |

The model outputs one of **three fixed labels** instead of free text; every message the user reads
comes from **templates in the app**. Teaching the model this behavior is part of the fine-tuning.

# The task: a voice command layer for a mobile app

## The product
Most mobile apps expose a set of actions: create, edit, search, delete, change settings...
Today the user reaches them by tapping through menus.

We want to add a **voice layer**: the user speaks a command in natural language,
and the app executes the corresponding action.

> 🎙️ "Crea un promemoria per domani alle 9"
> → the app calls `create_reminder(...)`

The language model does not chat and does not answer questions:
its only job is to translate a spoken request into a **call to one of the app's functions**.

## Why on-device?
- **Privacy**: the user's commands and data never leave the phone
- **Latency**: a command must feel instant (< 1 s)
- **Offline**: works without a connection
- **Cost**: zero inference cost per command, whatever the number of users

## Architecture

```
 🎙️ voice
   │
   ▼
 ┌──────────────────┐
 │ ASR (on-device)  │  speech → text
 └────────┬─────────┘
          ▼
 ┌──────────────────┐
 │ FunctionGemma    │  text → function call
 │ 270M + LoRA      │
 └────────┬─────────┘
          ▼
 ┌──────────────────┐
 │ Validator        │  schema check, resolve references,
 │ (plain code)     │  confirmation for destructive actions
 └────────┬─────────┘
          ▼
 ┌──────────────────┐        ┌───────────────────────┐
 │ App functions    │        │ Cloud / bigger model  │
 │ (executor)       │        │ for requests too      │
 └──────────────────┘        │ complex for on-device │
                             └───────────────────────┘
```

**Today's focus: the FunctionGemma box**, and how to make a 270M model reliable at it.

## Design principle: the model extracts, the app resolves

A 270M model is a specialist. We keep its job as small as possible:

| the model does | the app does (deterministic code) |
|---|---|
| choose the function | match names to real objects in the app (fuzzy search) |
| copy names and values as the user said them | turn "domani" or "la settimana scorsa" into real dates |
| map vague expressions to fixed values (enums) | check that values are valid |
| use placeholders for context ("questo", "quello che ho copiato") | know what is on screen or in the clipboard |
| decide that *no* function applies | handle the conversation around it |

Every decision we move into code is accuracy we don't need to train.

## Voice changes the input
The model never sees typed text, it sees **transcripts**:
- no punctuation, often lowercase: `crea un promemoria domani alle nove`
- numbers as words: `nove e mezza`, `dodici euro e cinquanta`
- hesitations and restarts: `ehm crea no anzi cerca le note su...`
- ASR mistakes on names and technical terms: `lora` → `l'ora`

Our dataset will have to look like this, not like clean written text.

## The use case: a voice layer for a video-notes app

The app turns YouTube videos into **atomic notes**: an LLM reads the transcript and extracts one note
per concept, each with a short explanation and the **timestamp** where the concept appears in the video.

Content is organized in three levels:

```
Collection                        e.g. "LLM fine-tuning"
 └── Video                        e.g. "LoRA explained"
      └── Note                    e.g. "LoRA freezes the base weights"  @ 02:15
```

- a **collection** groups videos by topic
- a **video** belongs to exactly one collection and contains many notes
- a **note** belongs to exactly one video and points to a moment in it

Videos are added through the normal UI (paste a URL). The **voice layer** handles navigation and
organization: the user speaks a command, and FunctionGemma turns it into a call to one of the app's functions.

### The functions

| level | function | arguments | what it does | example command |
|---|---|---|---|---|
| collection | `create_collection` | `name` | creates an empty collection | *"create a collection called speech models"* |
| collection | `open_collection` | `collection` | shows the videos in a collection | *"open the fine-tuning collection"* |
| collection | `delete_collection` | `collection` | deletes a collection and all its videos and notes ⚠️ | *"delete the old tests collection"* |
| video | `open_video` | `video` | shows the notes of a video | *"open the notes of the LoRA video"* |
| video | `move_video` | `video`, `collection` | moves a video to another collection | *"move this video to speech"* |
| video | `delete_video` | `video` | deletes a video and all its notes ⚠️ | *"delete this video"* |
| note | `search_notes` | `query`, `collection` (optional) | searches notes across videos | *"find notes about quantization"* |
| note | `open_note` | `note` | opens a single note | *"open the note about the rank"* |
| note | `play_note` | `note` | plays the video from the note's timestamp | *"play the part about attention"* |
| note | `delete_note` | `note` | deletes a single note ⚠️ | *"delete the note about the rank"* |

⚠️ Destructive action: the app always asks the user for confirmation before executing it.

`video` and `note` can also be **context references**: `@current` (what is open on screen) or `@latest` (the last added).

### Where we expect the model to struggle
- **Same verb, different level**: `open_collection` vs `open_video` vs `open_note`. The level is in the noun, not the verb.
- **Same verb, different target**: `play_video` (a video + optional time) vs `play_note` (a concept).
- **Find vs open**: `search_notes` (a list of results) vs `open_note` (one specific note).
- **Timestamps**: spoken times (*"minute twelve and a half"*, *"one hour five"*) must become a fixed format (`"12:30"`, `"1:05:00"`).
- **Optional arguments**: the model must not invent a `timestamp` or a `collection` the user never said.
- **Out-of-scope requests**: questions about the content (*"what does the video say about learning rates?"*) must not trigger any call.

In [3]:
def tool(name, description, properties, required):
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {"type": "object", "properties": properties, "required": required},
        },
    }

COLLECTION_REF = {"type": "string",
                  "description": "Name of the collection as said by the user, or '@current' for the collection currently open"}
VIDEO_REF = {"type": "string",
             "description": "Title of the video as said by the user, '@current' for the video currently open, "
                            "or '@latest' for the last added video"}
NOTE_REF = {"type": "string",
            "description": "Concept or title of the note as said by the user, or '@current' for the note currently open"}

APP_TOOLS = [
    # --- collection level
    tool("create_collection", "Creates a new empty collection of videos.", {
        "name": {"type": "string", "description": "Name of the new collection"},
    }, ["name"]),
    tool("open_collection", "Opens a collection and shows the videos it contains.", {
        "collection": COLLECTION_REF,
    }, ["collection"]),

    # --- video level
    tool("open_video", "Opens a video and shows the notes extracted from it.", {
        "video": VIDEO_REF,
    }, ["video"]),
    tool("play_video", "Plays a video, from the beginning or from a given time.", {
        "video": VIDEO_REF,
        "timestamp": {"type": "string",
                      "description": "Optional start time, formatted as MM:SS, or H:MM:SS for times over one hour"},
    }, ["video"]),
    tool("move_video", "Moves a video to another collection.", {
        "video": VIDEO_REF,
        "collection": {"type": "string", "description": "Name of the destination collection"},
    }, ["video", "collection"]),

    # --- note level
    tool("search_notes", "Searches notes by content, across all videos or within one collection.", {
        "query": {"type": "string", "description": "Words or concept to search for"},
        "collection": {"type": "string", "description": "Optional name of a collection to restrict the search"},
    }, ["query"]),
    tool("open_note", "Opens a single note.", {
        "note": NOTE_REF,
    }, ["note"]),
    tool("play_note", "Plays the source video of a note, starting from the moment where the note's concept appears.", {
        "note": NOTE_REF,
    }, ["note"]),
]

print([t["function"]["name"] for t in APP_TOOLS])

['create_collection', 'open_collection', 'open_video', 'play_video', 'move_video', 'search_notes', 'open_note', 'play_note']


In [7]:
# Inputs look like ASR transcripts: lowercase, no punctuation.
# expected = [] means the model should NOT call any function.
def call(fn, /, **arguments):
    return {"name": fn, "arguments": arguments}

PROBES = [
    # --- basic: clear command, one function
    ("basic", "create a collection called speech models",
     [call("create_collection", name="speech models")]),
    ("basic", "move the whisper video to speech models",
     [call("move_video", video="whisper", collection="speech models")]),
    ("basic", "find notes about quantization",
     [call("search_notes", query="quantization")]),

    # --- same verb, different level: the level is in the noun
    ("open_level", "open the fine tuning collection",
     [call("open_collection", collection="fine tuning")]),
    ("open_level", "open the lora video",
     [call("open_video", video="lora")]),
    ("open_level", "open the note about the rank",
     [call("open_note", note="the rank")]),

    # --- same verb, different target: a video vs a concept
    ("play_target", "play the lora video",
     [call("play_video", video="lora")]),
    ("play_target", "play the part where it explains attention",
     [call("play_note", note="attention")]),

    # --- find (a list) vs open (one note)
    ("find_vs_open", "show me all the notes about backpropagation",
     [call("search_notes", query="backpropagation")]),
    ("find_vs_open", "open the note about backpropagation",
     [call("open_note", note="backpropagation")]),

    # --- spoken time -> MM:SS or H:MM:SS
    ("timestamp", "play the lora video from minute twelve",
     [call("play_video", video="lora", timestamp="12:00")]),
    ("timestamp", "play the whisper video from three minutes and thirty seconds",
     [call("play_video", video="whisper", timestamp="3:30")]),
    ("timestamp", "start the transformers video at one hour and five minutes",
     [call("play_video", video="transformers", timestamp="1:05:00")]),

    # --- context references
    ("placeholder", "open this video",
     [call("open_video", video="@current")]),
    ("placeholder", "play the last video i added",
     [call("play_video", video="@latest")]),
    ("placeholder", "move this video to the speech collection",
     [call("move_video", video="@current", collection="speech")]),

    # --- optional argument given vs not given
    ("optional_args", "search for notes about attention in the transformers collection",
     [call("search_notes", query="attention", collection="transformers")]),

    # --- ASR-style noise
    ("asr_noise", "um open the uh the lora video",
     [call("open_video", video="lora")]),
    ("asr_noise", "open the collection no wait search notes about rag",
     [call("search_notes", query="rag")]),

    # --- required argument missing: should not invent one
    ("missing_arg", "move this video",
     []),
    ("missing_arg", "open the collection",
     []),

    # --- no function applies
    ("irrelevance", "what does the video say about learning rates",
     []),
    ("irrelevance", "what's the weather like tomorrow",
     []),
    ("irrelevance", "thanks that's all",
     []),
]

print(f"{len(PROBES)} probes")

24 probes


In [8]:
import pandas as pd

def ask_with(tools, user_request, max_new_tokens=128):
    messages = [
        {"role": "developer", "content": DEVELOPER_PROMPT},
        {"role": "user", "content": user_request},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=False)
    return raw, parse_calls(raw)


def to_seconds(ts):
    """'12:00' -> 720, '1:05:00' -> 3900. Returns None if not a valid MM:SS / H:MM:SS string."""
    try:
        parts = [int(p) for p in str(ts).strip().split(":")]
    except ValueError:
        return None
    if len(parts) == 2:
        return parts[0] * 60 + parts[1]
    if len(parts) == 3:
        return parts[0] * 3600 + parts[1] * 60 + parts[2]
    return None


def norm(v):
    """Light normalization: free text compared case-insensitively, without extra spaces."""
    if isinstance(v, str):
        return " ".join(v.strip().lower().split())
    return v


def same_args(pred, gold):
    if set(pred) != set(gold):          # missing or invented arguments -> wrong
        return False
    for k in gold:
        if k == "timestamp":
            if to_seconds(pred[k]) is None or to_seconds(pred[k]) != to_seconds(gold[k]):
                return False
        elif norm(pred[k]) != norm(gold[k]):
            return False
    return True


rows = []
for category, request, expected in PROBES:
    raw, calls = ask_with(APP_TOOLS, request)
    if not expected:
        correct_fn = exact = (len(calls) == 0)
    else:
        correct_fn = len(calls) == 1 and calls[0]["name"] == expected[0]["name"]
        exact = correct_fn and same_args(calls[0]["arguments"], expected[0]["arguments"])
    rows.append({
        "category": category,
        "request": request,
        "expected": expected,
        "predicted": calls if calls else raw.strip()[:60],
        "should_call": bool(expected),
        "made_call": bool(calls),
        "function_ok": correct_fn,
        "exact_match": exact,
    })

results = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 80)
results

NameError: name 'APP_TOOLS' is not defined

In [17]:
summary = (results.groupby("category")[["function_ok", "exact_match"]]
           .mean().mul(100).round(0)
           .assign(n=results.groupby("category").size()))
print(summary)

positives = results[results.should_call]
negatives = results[~results.should_call]
print(f"\nrelevance   (called when it should):     {positives.made_call.mean():.0%}")
print(f"irrelevance (no call when it shouldn't): {(~negatives.made_call).mean():.0%}")
print(f"exact match on positive probes:          {positives.exact_match.mean():.0%}")

               function_ok  exact_match  n
category                                  
asr_noise              0.0          0.0  2
basic                 33.0         33.0  3
find_vs_open         100.0         50.0  2
irrelevance           67.0         67.0  3
missing_arg           50.0         50.0  2
open_level            67.0          0.0  3
optional_args        100.0          0.0  1
placeholder            0.0          0.0  3
play_target          100.0         50.0  2
timestamp            100.0         33.0  3

relevance   (called when it should):     84%
irrelevance (no call when it shouldn't): 60%
exact match on positive probes:          21%


## Evaluation taxonomy (BFCL-style)

We follow the taxonomy of the **Berkeley Function Calling Leaderboard (BFCL)**:
[BFCL v3 on EvalScope](https://evalscope.readthedocs.io/en/latest/third_party/bfcl_v3.html)

BFCL splits its single-turn data into *non-live* (curated by the authors) and *live* (real user queries)
subsets. We merge them: what matters for us is the **type of behavior** being tested, not where the query comes from.
We also leave out the **multi-turn** subsets (stateful conversations over several turns) and the
**Java/JavaScript** subsets (calls in other languages' syntax): our voice layer handles one command at a time,
in FunctionGemma's own format.

| category | what is given to the model | expected behavior | example in our app |
|---|---|---|---|
| **simple** | one function | one call, correct arguments | only `create_note` available: *"crea una nota sulla lora"* → `create_note(title="lora")` |
| **multiple** | several functions | choose the right one, one call | all tools: *"collega la nota su lora a quella sulla quantizzazione"* → `link_notes(source="lora", target="quantizzazione")` |
| **parallel** | one function | several calls of it in one turn | *"tagga nlp le note su bert e lora"* → `add_tags(note="bert", tags="nlp")`, `add_tags(note="lora", tags="nlp")` |
| **parallel_multiple** | several functions | several different calls in one turn | *"crea una nota su rag e collegala a retrieval"* → `create_note(title="rag")`, `link_notes(source="rag", target="retrieval")` |
| **irrelevance** | functions that don't fit the request | **no call** | *"che tempo fa domani"* → no call |
| **relevance** | at least one function fits | **some call** (not a refusal or a text answer) | *"elimina la nota su bert"* → a call, not *"I cannot assist…"* |

### How each category is scored
- **simple, multiple, parallel, parallel_multiple**: AST matching. The call is parsed and compared with the expected one:
  function name, parameter names, types, and values (strings compared case-insensitively, whitespace normalized).
  For parallel categories, every expected call must be present.
- **irrelevance**: correct if the model makes no call.
- **relevance**: correct if the model makes a call.

> ⚠️ **irrelevance** and **relevance** pull in opposite directions: a model that never calls a function
> scores 100% on irrelevance and 0% on relevance. Always read them together.

![](https://sail-moe.oss-cn-hangzhou.aliyuncs.com/yunlin/images/evalscope/doc/bfcl_v3.jpg)

In [2]:
import asyncio
from ragas.metrics.collections import ToolCallAccuracy
from ragas.messages import AIMessage, HumanMessage, ToolCall

async def evaluate_tool_call_accuracy():
    # Define the conversation with tool calls
    user_input = [
        HumanMessage(content="What's the weather like in New York right now?"),
        AIMessage(
            content="The current temperature in New York is 75°F and it's partly cloudy.",
            tool_calls=[ToolCall(name="weather_check", args={"location": "New York"})],
        ),
        HumanMessage(content="Can you translate that to Celsius?"),
        AIMessage(
            content="Let me convert that to Celsius for you.",
            tool_calls=[
                ToolCall(
                    name="temperature_conversion", args={"temperature_fahrenheit": 75}
                )
            ],
        ),
    ]

    # Define expected tool calls
    reference_tool_calls = [
        ToolCall(name="weather_check", args={"location": "New York"}),
        ToolCall(name="temperature_conversion", args={"temperature_fahrenheit": 75}),
    ]

    # Evaluate
    metric = ToolCallAccuracy()
    result = await metric.ascore(
        user_input=user_input,
        reference_tool_calls=reference_tool_calls,
    )
    print(f"Tool Call Accuracy: {result.value}")

await evaluate_tool_call_accuracy()

Tool Call Accuracy: 1.0


Scoring Examples
Perfect match:

```
# All tools called correctly with correct arguments
Expected: [weather_check(location="Paris"), translate(text="hello")]
Got:      [weather_check(location="Paris"), translate(text="hello")]
Score: 1.0
```

Partial argument match:

```
# Some arguments incorrect
Expected: [search(query="python", limit=10, sort="date")]
Got:      [search(query="python", limit=10, sort="relevance")]
Score: 0.66 (2 out of 3 arguments match)
```

Wrong order (strict mode):

```
# Correct tools but wrong sequence
Expected: [search(...), filter(...)]
Got:      [filter(...), search(...)]
Score: 0.0 (sequence not aligned)
```

In [10]:
import json
import pandas as pd
from ragas.metrics.collections import ToolCallAccuracy
from ragas.messages import AIMessage, HumanMessage, ToolCall
from tqdm.auto import tqdm

with open("tool_calling_test.jsonl", encoding="utf-8") as f:
    test_rows = [json.loads(line) for line in f]

metric = ToolCallAccuracy()


async def eval_row(row):
    user, gold = row["messages"]
    should_call = "tool_calls" in gold
    raw, calls = ask_with(json.loads(row["tools"]), user["content"])

    if not should_call:
        # no-call rows: correct = the model made no call
        score = float(not calls)
    elif not calls:
        score = 0.0  # a call was expected and none was made
    else:
        result = await metric.ascore(
            user_input=[
                HumanMessage(content=user["content"]),
                AIMessage(
                    content=raw,
                    tool_calls=[ToolCall(name=c["name"], args=c["arguments"]) for c in calls],
                ),
            ],
            reference_tool_calls=[
                ToolCall(name=c["function"]["name"], args=c["function"]["arguments"])
                for c in gold["tool_calls"]
            ],
        )
        score = result.value

    return {
        "id": row["id"],
        "category": row["category"],
        "utterance": user["content"],
        "should_call": should_call,
        "made_call": bool(calls),
        "score": score,
        "raw": raw,
        "function": gold["tool_calls"][0]["function"]["name"] if should_call else None,
    }


results = []
for r in tqdm(test_rows, desc="evaluating"):
    results.append(await eval_row(r))
results = pd.DataFrame(results)

summary = results.groupby("category").agg(
    n=("score", "size"),
    mean_score=("score", "mean"),
    exact=("score", lambda s: (s == 1.0).mean()),
)
print(summary.round(3))

positives = results[results.should_call]
negatives = results[~results.should_call]
print(f"\ncalled when it should:        {positives.made_call.mean():.0%}")
print(f"no call when it shouldn't:    {(~negatives.made_call).mean():.0%}")
print(f"ragas tool call accuracy (simple): {positives.score.mean():.3f}")
simple = results[results.category == "simple"]
by_function = simple.groupby("function").agg(
    n=("score", "size"),
    made_call=("made_call", "mean"),
    mean_score=("score", "mean"),
    exact=("score", lambda s: (s == 1.0).mean()),
)
print(by_function.sort_values("exact").round(3))


evaluating:   4%|▍         | 25/600 [00:32<10:28,  1.09s/it]/home/lpozzi/Git/data-science-lectures/llm_lora/.venv/lib/python3.11/site-packages/ragas/metrics/collections/tool_call_accuracy/metric.py:135: UserWarning: Length mismatch: predicted tool calls (2) vs reference tool calls (1). Only the first 1 tool calls will be compared.
  warnings.warn(
evaluating: 100%|██████████| 600/600 [09:31<00:00,  1.05it/s]

                 n  mean_score  exact
category                             
missing_arg     50       0.500   0.50
not_supported   50       0.460   0.46
simple         500       0.324   0.32

called when it should:        74%
no call when it shouldn't:    48%
ragas tool call accuracy (simple): 0.324
                    n  made_call  mean_score  exact
function                                           
open_video         50       0.90        0.00   0.00
move_video         50       0.46        0.08   0.04
play_note          50       0.42        0.04   0.04
open_note          50       0.94        0.22   0.22
delete_video       50       0.78        0.28   0.28
delete_note        50       0.56        0.40   0.40
open_collection    50       0.92        0.48   0.48
delete_collection  50       1.00        0.48   0.48
search_notes       50       0.50        0.48   0.48
create_collection  50       0.90        0.78   0.78


In [11]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"test": "tool_calling_test.jsonl",
                                      "train": "tool_calling_train.jsonl"})
ds.push_to_hub("lopozz/videonotes-tool-calling", private=False)


Generating test split: 600 examples [00:00, 34502.09 examples/s]
Generating train split: 580 examples [00:00, 35431.06 examples/s]
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 93.99ba/s]
Processing Files (1 / 1): 100%|██████████| 46.4kB / 46.4kB, 38.7kB/s  
New Data Upload: 100%|██████████| 46.4kB / 46.4kB, 38.7kB/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 88.93ba/s]
Processing Files (1 / 1): 100%|██████████| 47.3kB / 47.3kB, 59.1kB/s  
New Data Upload: 100%|██████████| 47.3kB / 47.3kB, 59.1kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.70s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/lopozz/videonotes-tool-calling/commit/c6b737eb068e512c6f53ce2914dc9aff1b57b27a', commit_message='Upload dataset', commit_description='', oid='c6b737eb068e512c6f53ce2914dc9aff1b57b27a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/lopozz/videonotes-tool-calling', endpoint='https://huggingface.co', repo_type='dataset', repo_id='lopozz/videonotes-tool-calling'), pr_revision=None, pr_num=None)

# Training

## Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

## Unsloth

In [1]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # Can choose any sequence length!
fourbit_models = [
    # 4bit Gemma 3 dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-270m-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",
    # Function Gemma models
    "unsloth/functiongemma-270m-it",
    "unsloth/functiongemma-270m-it-unsloth-bnb-4bit",
    "unsloth/functiongemma-270m-it-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/functiongemma-270m-it",
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    load_in_16bit = True, # [NEW!] Enables 16bit LoRA
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

/home/lpozzi/Git/data-science-lectures/llm_lora/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/lpozzi/Git/data-science-lectures/llm_lora/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.10: Fast Gemma3 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA RTX 2000 Ada Generation Laptop GPU. Num GPUs = 1. Max memory: 7.625 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 236/236 [00:00<00:00, 626.47it/s]


Unsloth: compiling each transformer block as one region (Gemma3DecoderLayer).


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


[unsloth.import_fixes|WARNING]Unsloth: Ignoring an unusable torchao so LoRA can still be built (Found an incompatible version of torchao. Found version 0.14.1, but only versions above 0.16.0 are supported). Run `pip install --upgrade torchao` if you need torchao quantization.


## Train the model
Now let's train our model. We do 100 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [3]:
import json 
# # We split the dataset into some training and testing
# split_dataset = dataset.train_test_split(test_size = 50, shuffle = True, seed = 3407)
from datasets import load_dataset

dataset = load_dataset("lopozz/videonotes-tool-calling")  # add token=... if private

DEVELOPER_PROMPT = "You are a model that can do function calling with the following functions"

def to_text(example):
    messages = [{"role": "developer", "content": DEVELOPER_PROMPT}]
    for m in example["messages"]:
        m = {k: v for k, v in m.items() if v is not None}  # drop content / tool_calls set to None
        for c in m.get("tool_calls", []):
            args = c["function"]["arguments"]
            c["function"]["arguments"] = {k: v for k, v in args.items() if v is not None}
        messages.append(m)

    text = tokenizer.apply_chat_template(
        messages,
        tools=json.loads(example["tools"]),
        tokenize=False,
    ).removeprefix("<bos>")  # the trainer adds the BOS token again
    return {"text": text}

dataset = dataset.map(to_text)

import numpy as np


def exact_match_args(pred, ref):
    if not ref and not pred:
        return 1.0
    if not ref:
        return 0.0
    return sum(k in pred and str(pred[k]) == str(ref[k]) for k in ref) / len(ref)


def tool_call_accuracy(pred_calls, ref_calls):
    """Same scoring as ragas ToolCallAccuracy (strict order)."""
    if not pred_calls and not ref_calls:
        return 1.0
    if not pred_calls or not ref_calls:
        return 0.0
    if [c["name"] for c in pred_calls] != [c["name"] for c in ref_calls]:
        return 0.0
    return sum(exact_match_args(p["arguments"], r["arguments"]) for p, r in zip(pred_calls, ref_calls)) / len(ref_calls)


def preprocess_logits(logits, labels):
    return logits.argmax(-1)  # keep token ids, not vocab-sized logits

import re
import json

CALL_RE = re.compile(r"<start_function_call>call:([\w\.]+)\{(.*?)\}<end_function_call>", re.S)
ARG_RE = re.compile(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]+))", re.S)


def parse_value(raw):
    raw = raw.strip()
    try:
        return json.loads(raw)  # numbers, true/false, null
    except json.JSONDecodeError:
        return raw


def parse_calls(text):
    """Extract [{"name": ..., "arguments": {...}}] from FunctionGemma output.
    Flat arguments only: nested objects/arrays are not handled."""
    calls = []
    for name, args_str in CALL_RE.findall(text):
        args = {}
        for key, escaped, bare in ARG_RE.findall(args_str):
            args[key] = escaped if escaped or not bare else parse_value(bare)
        calls.append({"name": name, "arguments": args})
    return calls

def compute_metrics(eval_pred):
    preds, labels = eval_pred.predictions, eval_pred.label_ids
    preds, labels = preds[:, :-1], labels[:, 1:]  # logits at t predict token t+1
    scores = []
    for p, l in zip(preds, labels):
        m = l != -100  # response tokens only
        pred_calls = parse_calls(tokenizer.decode(p[m], skip_special_tokens=False))
        ref_calls = parse_calls(tokenizer.decode(l[m], skip_special_tokens=False))
        scores.append(tool_call_accuracy(pred_calls, ref_calls))
    return {"tool_call_accuracy": float(np.mean(scores))}

from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["test"], # For evaluation!
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 50,
        eval_steps = 25,
        eval_strategy = "steps",
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        # report_to = "none",
        report_to = "trackio",
        project = "videonotes-tool-calling",   # groups the runs in the dashboard
        run_name = "lr2e-4_r16",               # one name per experiment
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=8): 100%|██████████| 600/600 [01:09<00:00,  8.59 examples/s]


We also mask all system and user instructions so our goal is to force the finetune to actually respond with the correct function calls. If not, the finetune might just learn the format, and not learn anything else.

In [4]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

Map: 100%|██████████| 600/600 [00:00<00:00, 1155.52 examples/s]


We print the full un-masked string:

In [5]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

"<bos><start_of_turn>developer\nYou are a model that can do function calling with the following functions<start_function_declaration>declaration:create_collection{description:<escape>Creates a new empty collection of videos.<escape>,parameters:{properties:{name:{description:<escape>Name of the new collection<escape>,type:<escape>STRING<escape>}},required:[<escape>name<escape>],type:<escape>OBJECT<escape>}}<end_function_declaration><start_function_declaration>declaration:open_collection{description:<escape>Opens a collection and shows the videos it contains.<escape>,parameters:{properties:{collection:{description:<escape>Name of the collection as said by the user<escape>,type:<escape>STRING<escape>}},required:[<escape>collection<escape>],type:<escape>OBJECT<escape>}}<end_function_declaration><start_function_declaration>declaration:delete_collection{description:<escape>Deletes a collection, together with all its videos and notes.<escape>,parameters:{properties:{collection:{description:<e

See if we did the masking correctly - "-" is the mask

In [6]:
[tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, "-")]

['--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------<start_function_call>call:search_notes{query:<escape>mental models<escape>}<end_function_call><start_function_response>']

In [11]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA RTX 2000 Ada Generation Laptop GPU. Max memory = 7.625 GB.
0.535 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [7]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 580 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 3,796,992 of 271,895,168 (1.40% trained)


* Trackio project initialized: videonotes-tool-calling
* Trackio metrics logged to: /home/lpozzi/.cache/huggingface/trackio
* NVIDIA GPU detected, enabling automatic GPU metrics logging
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: lr2e-4_r16


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss,Tool Call Accuracy
25,0.088226,0.080749,0.862500
50,0.025507,0.042449,0.906667


Filter: 100%|██████████| 600/600 [00:01<00:00, 488.95 examples/s]


* Run finished. Uploading logs to Trackio (please wait...)


In [8]:
def model_init(trial=None):
    model, _ = FastLanguageModel.from_pretrained(...)
    r = trial.suggest_categorical("r", [8, 16, 32]) if trial else 16
    return FastLanguageModel.get_peft_model(model, r=r, lora_alpha=r, ...)

def hp_space(trial):
    return {"learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)}

trainer = SFTTrainer(model_init=model_init, ..., compute_metrics=compute_metrics)
best = trainer.hyperparameter_search(
    hp_space=hp_space,
    compute_objective=lambda m: m["eval_tool_call_accuracy"],
    direction="maximize",
    n_trials=1,
    backend="optuna",
)
print(best.hyperparameters)


SyntaxError: positional argument follows keyword argument (1588050582.py, line 4)

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## Inference
Let's run the model via Unsloth native inference for the first message which should create a calendar reminder

In [6]:
import json

row = dataset["test"][1]
messages = [
    {"role": "developer", "content": DEVELOPER_PROMPT},
    row["messages"][0],  # user turn only; messages[1] is the gold answer
]
text = tokenizer.apply_chat_template(
    messages,
    tools=json.loads(row["tools"]),
    tokenize=False,
    add_generation_prompt=True,  # must add for generation
).removeprefix("<bos>")

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    max_new_tokens=128,
    streamer=TextStreamer(tokenizer, skip_prompt=False),
    do_sample=False,
)
print("\ngold:", row["messages"][1])


<bos><start_of_turn>developer
You are a model that can do function calling with the following functions<start_function_declaration>declaration:create_collection{description:<escape>Creates a new empty collection of videos.<escape>,parameters:{properties:{name:{description:<escape>Name of the new collection<escape>,type:<escape>STRING<escape>}},required:[<escape>name<escape>],type:<escape>OBJECT<escape>}}<end_function_declaration><start_function_declaration>declaration:open_collection{description:<escape>Opens a collection and shows the videos it contains.<escape>,parameters:{properties:{collection:{description:<escape>Name of the collection as said by the user<escape>,type:<escape>STRING<escape>}},required:[<escape>collection<escape>],type:<escape>OBJECT<escape>}}<end_function_declaration><start_function_declaration>declaration:delete_collection{description:<escape>Deletes a collection, together with all its videos and notes.<escape>,parameters:{properties:{collection:{description:<esc

## Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("functiongemma_lora")  # Local saving
tokenizer.save_pretrained("functiongemma_lora")
# model.push_to_hub("your_name/functiongemma_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/functiongemma_lora", token = "YOUR_HF_TOKEN") # Online saving

### Saving to float16
We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("functiongemma_finetune_16bit", tokenizer, save_method = "merged_16bit")
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/functiongemma_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False:
    model.save_pretrained_merged("functiongemma_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/functiongemma_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("functiongemma_lora")
    tokenizer.save_pretrained("functiongemma_lora")
if False: # Pushing to HF Hub
    model.push_to_hub("HF_USERNAME/functiongemma_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/functiongemma_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [ ]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "functiongemma_finetune",
        tokenizer,
        quantization_method = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "HF_ACCOUNT/functiongemma_finetune",
        tokenizer,
        quantization_method = "Q8_0", # Only Q8_0, BF16, F16 supported
        token = "YOUR_HF_TOKEN",
    )

# TODO
- how to find hyperparameters
- where to save run stats
- where to save results for error analysis
- possible to evaluate on task metrics?